# Amazon ML Challenge 2026 — full pipeline on Kaggle

Runs the same code as the local sample, on the full dataset.
Inputs (attach with **Add Input** in the right-hand panel):
- your private **dataset** with the challenge zip (train/test TSVs + validate_submission.py)
- your private **code** dataset (the `code/` folder)

Outputs land in `/kaggle/working/` → download from the **Output** tab.

## 1. Install packages (needs Settings → Internet: On)

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 unidecode==1.4.0
import polars, rapidfuzz, lightgbm
print("polars", polars.__version__, "| rapidfuzz", rapidfuzz.__version__, "| lightgbm", lightgbm.__version__)

## 2. Find the data and code folders
Kaggle mounts every attached dataset under `/kaggle/input/<name>/`; we search for them so the exact names don't matter.

In [ ]:
from pathlib import Path
import subprocess, sys, os

inp = Path("/kaggle/input")
data_root = next(p.parent for p in inp.rglob("train_source1.tsv") if p.parent.name == "train").parent
code_dir = next(p.parent for p in inp.rglob("run_pipeline.py"))
validator = next(inp.rglob("validate_submission.py"), None)
WORK = Path("/kaggle/working/work")
print("data:", data_root, "\ncode:", code_dir, "\nvalidator:", validator)
!ls -la {data_root}/train {data_root}/test
!free -g; nproc; df -h /kaggle/working | tail -1

## 3. Run the pipeline
Each stage prints progress. `--train-fraction 0.4` fits the model on 40% of the training entities
to stay inside Kaggle's 30 GB RAM (blocking/features still use all data). Raise it if memory allows.

In [ ]:
def run(*stages, extra=""):
    cmd = f"python {code_dir}/run_pipeline.py --data {data_root} --work {WORK} {' '.join(stages)} {extra}"
    print(">>", cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    assert rc == 0, f"stage failed: {stages}"

run("learn", "normalize-train")

In [ ]:
run("featurize-train")   # prints BLOCKING RECALL per country

In [ ]:
run("train", extra="--train-fraction 0.4")   # prints CV macro F0.5 and the chosen threshold

In [ ]:
run("normalize-test", "featurize-test", "predict")

## 4. Validate and publish the outputs

In [ ]:
import shutil
out = WORK / "output"
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    shutil.copy(out / f, Path("/kaggle/working") / f)
if validator:
    !python {validator} -m /kaggle/working/matching_results.tsv -c /kaggle/working/candidate_pairs.tsv -t {data_root}/test
!head -3 /kaggle/working/matching_results.tsv

## 5. Free space (Kaggle keeps only 20 GB of output)
The cache is large and not needed after predicting. Keep the model for later reuse.

In [ ]:
shutil.rmtree(WORK / "cache", ignore_errors=True)
!du -sh /kaggle/working/*